# Model C — U-Net (ResNet-34 encoder) segmentation
**Owner:** Kumar Kshitij

U-Net trained on box-derived damage masks (RDD2022 has no pixel masks — state this limitation in the viva). 15 epochs at 512 px ≈ 1.5–2 h on a T4; use `--size 384` if short of time.

**Before running:** Kaggle → *Settings* → Accelerator **GPU T4 x2**, Internet **On**. *Add Input* → dataset **aliabdelmenam/rdd-2022** (the same RDD2022 `RDD_SPLIT` used in Phase 2). Then *Run All*.
The final cell zips `results/` → download it from the Output panel and put it into the repo's `results/` folder.

In [ ]:
import os, subprocess, sys
# Option A (recommended): the final code is pushed to the team repo
if not os.path.exists('repo'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/lakshya1333/Road-damage-detection-deep-learning-project', 'repo'], check=True)
# Option B: if you uploaded the code folder as a Kaggle dataset instead, copy it:  !cp -r /kaggle/input/<your-code-dataset> repo
os.chdir('repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.2', 'segmentation-models-pytorch>=0.3'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Shared manifest: identical split + class mapping for all four models (check the printed class mapping!)
!python -m src.data.build_manifest --out results/data --plot

In [ ]:
!python -m src.models.train_unet --manifest results/data --out results/unet --epochs 15 --size 512 --batch 12

In [ ]:
from IPython.display import Image, display
display(Image('results/unet/unet_samples.png'))

In [ ]:
import shutil, glob, os
# drop large weights from the archive except the best checkpoint
shutil.make_archive('/kaggle/working/results_unet', 'zip', 'results')
print(open(sorted(glob.glob('results/*/metrics.json'))[0]).read()[:3000])